In [33]:
import pandas as pd

eval_df = pd.read_json('/Users/pariidan/Documents/python_projects/RAG_Langchain/data/evaluation/rag_strategy_responses.json')

In [34]:
def parse_response(row):
    # Remove query from the beginning of response to avoid partial matches
    if row['response'].startswith(row['query']):
        row['response'] = row['response'][len(row['query']):].lstrip()
    return row

# Apply the function to each row
eval_df = eval_df.apply(parse_response, axis=1)

In [ ]:
# eval_df = eval_df.loc[eval_df['error'].isna()]
# # Convert to list of dictionaries (records format)
# eval_df['timestamp'] = eval_df['timestamp'].apply(str)
# import json
# list_format = eval_df.to_dict('records')

# # Save back in the original format your code expects
# with open('/Users/pariidan/Documents/python_projects/RAG_Langchain/data/evaluation/rag_strategy_responses.json', 'w', encoding='utf-8') as f:
#     json.dump(list_format, f, ensure_ascii=False, indent=2)

In [37]:
eval_df

,strategy,query,response,retrieved_context,timestamp,response_time,use_prefiltering,error,use_reranker
0,original_prompt,What work activities did Cristina and I discus...,"In our WhatsApp messages, we discussed work ac...",Source: Cristina\nDate Range: 2024-02-21 - 202...,2025-08-11 00:29:56.242000,31.468894,False,None,False
1,original_prompt,Which musicians did I mention in my chats and ...,"In your chats, you mentioned liking Amy Wineho...",Source: McGregor one love\nDate Range: 2020-01...,2025-08-11 00:30:27.712000,16.689219,False,None,False
2,original_prompt,Show me discussions from my messages about gym...,"In your messages, you discussed gym and workou...",Source: Instagram user 3\nDate Range: 2018-10-...,2025-08-11 00:30:44.402000,20.937831,False,None,False
3,original_prompt,What did Teodor Lunugu and I talk about regard...,"In our conversations, Teodor Lunugu and I disc...",,2025-08-11 00:31:05.341000,10.914283,False,None,False
4,original_prompt,What daily activities did I tell Cristina abou...,"In our recent messages, you told Cristina abou...",Source: Cristina\nDate Range: 2022-01-15 - 202...,2025-08-11 00:31:16.257000,32.587314,False,None,False
...,...,...,...,...,...,...,...,...,...
120,verification_approach,What events did I plan to watch together with ...,You planned to watch a movie together with fri...,Source: Тъпанари and Krisi\nDate Range: 2022-0...,2025-08-13 17:22:50.310957,57.669203,False,None,True
121,verification_approach,Did I play any games with other people recentl...,You mentioned playing a game of Among Us with ...,None,2025-08-13 17:23:47.993645,5.711200,False,None,True
122,verification_approach,What homework projects am I doing at my Data S...,"You are working on a project called ""IRTM Proj...",Source: Puișor♥️🐢\nDate Range: 2024-04-09 - 20...,2025-08-13 17:23:53.710510,1798.165342,False,None,True
123,verification_approach,Discussing grades on recent exams and projects,"You mentioned a graded document called ""Graded...",None,2025-08-13 17:53:51.891966,6.749635,False,None,True


In [38]:
# Define test queries to filter by
test_queries = [
    "Which stocks did I talk to investing in with others",
    "What events did I plan to watch together with my friends",
    "Did I play any games with other people recently? What games and which people?",
    "What homework projects am I doing at my Data Science University, who do I talk to about it?",
    "Discussing grades on recent exams and projects"
]

# Filter eval_df to only include test queries
filtered_df = eval_df[eval_df['query'].isin(test_queries)]

# Simple version - group by query to compare strategies
for query in test_queries:
    if query in filtered_df['query'].values:
        print(f"\n{'='*70}")
        print(f"🔍 QUERY: {query}")
        print('='*70)
        
        query_data = filtered_df[filtered_df['query'] == query]
        
        for strategy in query_data['strategy'].unique():
            print(f"\n🎯 STRATEGY: {strategy.upper()}")
            print('-'*40)
            
            strategy_responses = query_data[query_data['strategy'] == strategy]
            
            for idx, (_, row) in enumerate(strategy_responses.iterrows(), 1):
                if row['retrieved_context'] is not None:
                    if row['use_reranker']:
                        print('With Rerenker:')
                    print(f"\n📝 Response #{idx}:")
                    print(f"{row['response']}\n")
                else:
                    print(f"\n⚠️ No retrieved context for Response #{idx}")
                    print(f"📝 Response #{idx}:")
                    print(f"{row['response']}\n")
            
            print('-'*40)
    else:
        print(f"\n❌ Query not found in data: {query}")


🔍 QUERY: Which stocks did I talk to investing in with others

🎯 STRATEGY: ORIGINAL_PROMPT
----------------------------------------

📝 Response #1:
In the conversation with Mihai, you discussed investing in AI coins, with Mihai suggesting to "căută pe Google și a să vezi" and mentioning "Crypto AI Coins" on Ethereum. Mihai also said, "Cel mai bine ar fi AI coins," indicating you were considering this type of investment.

With Rerenker:

📝 Response #2:
In the chat, you discussed investing in stocks like "The Vanguard S&P 500 UCITS ETF" and mentioned "Voo" (likely referring to Vanguard's ETF). You also talked about Tesla, with Dan NL mentioning he bought 500 worth of Tesla. Additionally, there were discussions about investing in "dist" (possibly short for "distrito" or a stock ticker) and "ETFs on Revolut."

----------------------------------------

🎯 STRATEGY: CONDENSED_QUERY_CONTEXT
----------------------------------------

📝 Response #1:
In the conversation with Dan, you discussed inv

Strategy Notes :

DONT_KNOW_SCENARIOS:
 
    - can say no information even if it is available

CRITIQUE_AND_REVISION:

    - was able to correctly reasses and mention specific names (like in the stock example)
    - also hallucinate


CONTEXT_AWARENESS:
    - was also able to correctly identify stocks discussed
    - no hallucinations
    - more details than the rest, more infromative seemingly
    - did not know response een if it was in there in at least one case

ORIGINAL_PROMPT/ CONDENSED VERISON:
    - seems to misunderstand things like stock names or movie names (i.e hallucinate as different things and treat them as that)



Nice Prompts :

    - stocks question ( maybe more detail/direction)
    - question about a specfic person ( like Donald Trump or Elon Musk)
    - find a super specific question ( like something I was talking about buying like tickets to Travis Scott concert), ask how much they were
    - Something about my bachelor thesis, like what topic it was about etc.
    - question about how expeinsive my hotel was when I went to greece last summer, (quote with exact numbers from the tikkies)
    - offers great detailed reposne : " What crypto coins did my friends suggest buying?"
    